# 08 — Validate K=7 Aggregation Outputs and Build Reporting Files

This notebook checks the outputs from Notebook 07 and produces reporting-friendly files:

- named full profiles
- enriched ward profile
- ward and MSOA map-ready files
- North West ward subset
- plain-English K=7 cluster interpretation key

It is deliberately separate from Notebook 07: Notebook 07 builds the data, Notebook 08 validates and prepares it for use.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

## 1. Paths and configuration

In [2]:
NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name.lower() == "notebooks":
    PROJECT_DIR = NOTEBOOK_DIR.parent
else:
    PROJECT_DIR = NOTEBOOK_DIR

DATA_PROCESSED = PROJECT_DIR / "data" / "processed"
DATA_GEOGRAPHY = PROJECT_DIR / "data" / "geography"
OUTPUT_DIR = DATA_PROCESSED / "aggregations_v1"
K_OUTPUT_DIR = DATA_PROCESSED / "k_comparison_outputs_v1"
REPORT_DIR = K_OUTPUT_DIR / "reports"

K = 7

WARD_LOOKUP_PATH = DATA_GEOGRAPHY / "oa21_to_wd25_lad25_eng_wal(may25).csv"

print("Project folder:", PROJECT_DIR)
print("Processed folder:", DATA_PROCESSED)
print("Aggregation output folder:", OUTPUT_DIR)
print("K comparison output folder:", K_OUTPUT_DIR)
print("Report folder:", REPORT_DIR)

Project folder: c:\Users\keena\Documents\Electoral_Tribes
Processed folder: c:\Users\keena\Documents\Electoral_Tribes\data\processed
Aggregation output folder: c:\Users\keena\Documents\Electoral_Tribes\data\processed\aggregations_v1
K comparison output folder: c:\Users\keena\Documents\Electoral_Tribes\data\processed\k_comparison_outputs_v1
Report folder: c:\Users\keena\Documents\Electoral_Tribes\data\processed\k_comparison_outputs_v1\reports


## 2. Helper functions

In [3]:
CLUSTER_NAMES = {
    0: "Student & Transient Youth",
    1: "Rooted Older Homeowners",
    2: "Stable Suburban Professionals",
    3: "Cosmopolitan Young Professional Core",
    4: "Settled Working Families / Skilled Trades Suburbs",
    5: "Settled Diverse Urban Communities",
    6: "Post-Industrial Estates / Deprived Working Communities",
}


def norm_col(col: str) -> str:
    col = str(col).strip().upper()
    return re.sub(r"[^A-Z0-9]", "", col)


def find_col(df: pd.DataFrame, candidates=None, patterns=None, required=True, label="column"):
    candidates = candidates or []
    patterns = patterns or []
    norm_map = {norm_col(c): c for c in df.columns}

    for cand in candidates:
        key = norm_col(cand)
        if key in norm_map:
            return norm_map[key]

    for pattern in patterns:
        rx = re.compile(pattern)
        for normed, original in norm_map.items():
            if rx.search(normed):
                return original

    if required:
        raise ValueError(
            f"Could not find {label}. Candidates={candidates}; patterns={patterns}; "
            f"available={list(df.columns)}"
        )
    return None


def add_cluster_names(df):
    df = df.copy()
    df["dominant_cluster_name"] = df["dominant_cluster"].map(CLUSTER_NAMES)
    df["second_cluster_name"] = df["second_cluster"].map(CLUSTER_NAMES)

    for cid, name in CLUSTER_NAMES.items():
        share_col = f"cluster_{cid}_share"
        pop_col = f"cluster_{cid}_population"
        safe_name = re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_").lower()

        if share_col in df.columns:
            df[f"{safe_name}_share"] = df[share_col]
        if pop_col in df.columns:
            df[f"{safe_name}_population"] = df[pop_col]

    return df


def check_cluster_shares(df, k=7):
    share_cols = [f"cluster_{i}_share" for i in range(k)]
    available = [c for c in share_cols if c in df.columns]

    if not available:
        return pd.Series(dtype=float)

    check = df[available].sum(axis=1)
    return check.describe()


def pct(x):
    if pd.isna(x):
        return ""
    if isinstance(x, str):
        return x
    return f"{x:.1%}"


def find_existing_file(filename, search_dirs):
    for d in search_dirs:
        path = d / filename
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Could not find {filename}. Searched: {[str(d) for d in search_dirs]}"
    )

## 3. Load full profile outputs

This expects Notebook 07 to have produced the `k7_*_full_profile_v1.csv` files.

In [4]:
profile_files = {
    "oa_base": OUTPUT_DIR / f"k{K}_oa_geo_cluster_base_v1.csv",
    "lsoa21": OUTPUT_DIR / f"k{K}_lsoa21_full_profile_v1.csv",
    "msoa21": OUTPUT_DIR / f"k{K}_msoa21_full_profile_v1.csv",
    "ward25": OUTPUT_DIR / f"k{K}_ward25_full_profile_v1.csv",
    "lad23": OUTPUT_DIR / f"k{K}_lad23_full_profile_v1.csv",
    "lad25": OUTPUT_DIR / f"k{K}_lad25_full_profile_v1.csv",
}

data = {}

for name, path in profile_files.items():
    if path.exists():
        data[name] = pd.read_csv(path, low_memory=False)
        print(name, data[name].shape, "loaded from", path.name)
    else:
        print(name, "missing:", path)

required = ["lsoa21", "msoa21", "ward25", "lad25"]
missing_required = [name for name in required if name not in data]

if missing_required:
    raise FileNotFoundError(f"Missing required full profile outputs: {missing_required}")

oa_base (188880, 16) loaded from k7_oa_geo_cluster_base_v1.csv
lsoa21 (33755, 159) loaded from k7_lsoa21_full_profile_v1.csv
msoa21 (6856, 159) loaded from k7_msoa21_full_profile_v1.csv
ward25 (7572, 161) loaded from k7_ward25_full_profile_v1.csv
lad23 (296, 159) loaded from k7_lad23_full_profile_v1.csv
lad25 (318, 159) loaded from k7_lad25_full_profile_v1.csv


## 4. Validate cluster-share sums and population totals

In [5]:
print("Cluster-share sum checks")
for name, df in data.items():
    if name == "oa_base":
        continue

    print("\\n" + name)
    print(check_cluster_shares(df, K))

print("\\nPopulation totals")
for name, df in data.items():
    if "population" in df.columns:
        print(name, df["population"].sum())

Cluster-share sum checks
\nlsoa21
count    3.375500e+04
mean     1.000000e+00
std      7.231598e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64
\nmsoa21
count    6.856000e+03
mean     1.000000e+00
std      1.416254e-16
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64
\nward25
count    7.572000e+03
mean     1.000000e+00
std      1.384744e-16
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64
\nlad23
count    2.960000e+02
mean     1.000000e+00
std      2.386421e-16
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64
\nlad25
count    3.180000e+02
mean     1.000000e+00
std      2.371181e-16
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000

## 5. Ensure named versions exist

Notebook 07 already adds names. This section is defensive: it refreshes names and saves named versions consistently.

In [6]:
named_outputs = {}

for name, df in data.items():
    if name == "oa_base":
        continue

    named = add_cluster_names(df)
    named_outputs[name] = named

    out_path = OUTPUT_DIR / f"k{K}_{name}_named_full_profile_v1.csv"
    named.to_csv(out_path, index=False)
    print("Saved:", out_path.name)

Saved: k7_lsoa21_named_full_profile_v1.csv
Saved: k7_msoa21_named_full_profile_v1.csv
Saved: k7_ward25_named_full_profile_v1.csv
Saved: k7_lad23_named_full_profile_v1.csv
Saved: k7_lad25_named_full_profile_v1.csv


## 6. Ward validation and enriched ward file

The ward file must contain `LAD25CD` and `LAD25NM`. If those columns are missing because an older version of Notebook 07 was run, this cell patches them from the ward lookup.

In [7]:
ward = named_outputs["ward25"].copy()

required_ward_cols = ["LAD25CD", "LAD25NM", "WD25CD", "WD25NM"]
missing = [c for c in required_ward_cols if c not in ward.columns]

if missing:
    print("Ward file is missing LAD columns; patching from ward lookup:", missing)

    ward_lookup_raw = pd.read_csv(WARD_LOOKUP_PATH, low_memory=False)

    ward_cols = {
        "WD25CD": find_col(
            ward_lookup_raw,
            ["WD25CD", "WARD25CD", "Electoral Ward 2025 Code", "Ward Code"],
            patterns=[r"^WD25CD$", r"WARD.*2025.*CODE", r"ELECTORALWARD.*CODE"],
            label="WD25CD in ward lookup"
        ),
        "LAD25CD": find_col(
            ward_lookup_raw,
            ["LAD25CD", "Local Authority District 2025 Code", "LAD Code"],
            patterns=[r"^LAD25CD$", r"LOCALAUTHORITY.*2025.*CODE", r"LAD.*2025.*CODE"],
            label="LAD25CD in ward lookup"
        ),
        "LAD25NM": find_col(
            ward_lookup_raw,
            ["LAD25NM", "Local Authority District 2025 Name", "LAD Name"],
            patterns=[r"^LAD25NM$", r"LOCALAUTHORITY.*2025.*NAME", r"LAD.*2025.*NAME"],
            label="LAD25NM in ward lookup"
        ),
    }

    ward_lad_lookup = (
        ward_lookup_raw[list(ward_cols.values())]
        .rename(columns={v: k for k, v in ward_cols.items()})
        .drop_duplicates()
    )

    for col in ["WD25CD", "LAD25CD", "LAD25NM"]:
        ward_lad_lookup[col] = ward_lad_lookup[col].astype(str).str.strip()

    ward["WD25CD"] = ward["WD25CD"].astype(str).str.strip()

    duplicate_wards = (
        ward_lad_lookup
        .groupby("WD25CD")
        .size()
        .reset_index(name="n")
        .query("n > 1")
    )

    if len(duplicate_wards) > 0:
        display(duplicate_wards.head(20))
        raise ValueError("Some WD25CD values map to more than one LAD25. Review lookup before patching.")

    ward = (
        ward
        .drop(columns=["LAD25CD", "LAD25NM"], errors="ignore")
        .merge(ward_lad_lookup, on="WD25CD", how="left", validate="many_to_one")
    )

# Reorder key geography fields.
front_cols = ["LAD25CD", "LAD25NM", "WD25CD", "WD25NM"]
other_cols = [c for c in ward.columns if c not in front_cols]
ward = ward[front_cols + other_cols]

ward["is_mixed_ward"] = ward["dominant_cluster_share"] < 0.40
ward["is_clear_dominant_ward"] = ward["dominant_cluster_share"] >= 0.60
ward["is_highly_fragmented"] = ward["cluster_fragmentation_index"] >= 0.70

print("Missing LAD25CD:", ward["LAD25CD"].isna().sum())
print("Missing LAD25NM:", ward["LAD25NM"].isna().sum())

ward_enriched_path = OUTPUT_DIR / f"k{K}_ward25_named_full_profile_v1_enriched.csv"
ward.to_csv(ward_enriched_path, index=False)

print("Saved:", ward_enriched_path.name)

Missing LAD25CD: 0
Missing LAD25NM: 0
Saved: k7_ward25_named_full_profile_v1_enriched.csv


## 7. Map-ready files

These are slim outputs intended for joining to boundary files in QGIS, GeoPandas or another mapping workflow.

In [8]:
ward_map_cols = [
    "LAD25CD", "LAD25NM", "WD25CD", "WD25NM",
    "population", "oa_count",
    "dominant_cluster", "dominant_cluster_name", "dominant_cluster_share",
    "second_cluster", "second_cluster_name", "second_cluster_share",
    "cluster_fragmentation_index",
    "is_mixed_ward", "is_clear_dominant_ward", "is_highly_fragmented",
]

ward_map = ward[[c for c in ward_map_cols if c in ward.columns]].copy()
ward_map.to_csv(OUTPUT_DIR / f"k{K}_ward25_map_ready_v1.csv", index=False)

msoa = named_outputs["msoa21"].copy()

msoa_map_cols = [
    "MSOA21CD", "MSOA21NM",
    "population", "oa_count",
    "dominant_cluster", "dominant_cluster_name", "dominant_cluster_share",
    "second_cluster", "second_cluster_name", "second_cluster_share",
    "cluster_fragmentation_index",
]

msoa_map = msoa[[c for c in msoa_map_cols if c in msoa.columns]].copy()
msoa_map.to_csv(OUTPUT_DIR / f"k{K}_msoa21_map_ready_v1.csv", index=False)

print("Saved map-ready ward and MSOA files.")

Saved map-ready ward and MSOA files.


## 8. North West ward subset

This creates a North West working file from the LAD25 names.

In [9]:
north_west_lads = [
    # Cheshire
    "Cheshire East", "Cheshire West and Chester", "Halton", "Warrington",

    # Cumbria
    "Cumberland", "Westmorland and Furness",

    # Greater Manchester
    "Bolton", "Bury", "Manchester", "Oldham", "Rochdale", "Salford",
    "Stockport", "Tameside", "Trafford", "Wigan",

    # Lancashire / unitary / districts
    "Blackburn with Darwen", "Blackpool", "Burnley", "Chorley", "Fylde",
    "Hyndburn", "Lancaster", "Pendle", "Preston", "Ribble Valley",
    "Rossendale", "South Ribble", "West Lancashire", "Wyre",

    # Merseyside
    "Knowsley", "Liverpool", "Sefton", "St. Helens", "Wirral",
]

ward_nw = ward[ward["LAD25NM"].isin(north_west_lads)].copy()
ward_nw["north_west_subset"] = True

ward_nw["boundary_note"] = np.where(
    ward_nw["LAD25NM"].eq("Sefton"),
    "Sefton changed for 2026; WD25 is not suitable for final 2026/2027 ward mapping.",
    "WD25 used as current working geography."
)

ward_nw_path = OUTPUT_DIR / f"k{K}_north_west_ward25_full_profile_v1.csv"
ward_nw.to_csv(ward_nw_path, index=False)

print("North West wards:", len(ward_nw))
print("Saved:", ward_nw_path.name)

North West wards: 825
Saved: k7_north_west_ward25_full_profile_v1.csv


## 9. Build K=7 plain-English cluster interpretation key

This uses:

- `k7_cluster_profile_summaries_v1.csv`
- `k7_cluster_defining_features_v1.csv`
- `k7_cluster_feature_means_v1.csv`
- `k7_cluster_age_band_shares_v1.csv`

The notebook searches common project folders for these files, then saves:

`k7_cluster_interpretation_key_v1.csv`

In [10]:
summary_search_dirs = [
    REPORT_DIR,
    K_OUTPUT_DIR,
    DATA_PROCESSED,
    OUTPUT_DIR,
    PROJECT_DIR,
]

summary_path = find_existing_file(f"k{K}_cluster_profile_summaries_v1.csv", summary_search_dirs)
defining_path = find_existing_file(f"k{K}_cluster_defining_features_v1.csv", summary_search_dirs)
means_path = find_existing_file(f"k{K}_cluster_feature_means_v1.csv", summary_search_dirs)
age_path = find_existing_file(f"k{K}_cluster_age_band_shares_v1.csv", summary_search_dirs)

summaries = pd.read_csv(summary_path)
defining = pd.read_csv(defining_path)
means = pd.read_csv(means_path)
age = pd.read_csv(age_path)

print("Loaded:")
print(summary_path)
print(defining_path)
print(means_path)
print(age_path)

Loaded:
c:\Users\keena\Documents\Electoral_Tribes\data\processed\k7_cluster_profile_summaries_v1.csv
c:\Users\keena\Documents\Electoral_Tribes\data\processed\k7_cluster_defining_features_v1.csv
c:\Users\keena\Documents\Electoral_Tribes\data\processed\k7_cluster_feature_means_v1.csv
c:\Users\keena\Documents\Electoral_Tribes\data\processed\k7_cluster_age_band_shares_v1.csv


In [11]:
PROFILE_TEXT = {
    0: {
        "interpretation": "A small but very sharply defined student and short-term private-rental cluster. It is dominated by young adults, full-time students, recent residents, flats and private renting. It should be kept separate because folding it into a general urban/professional group would contaminate the model.",
        "confidence": "High — very distinctive, though small.",
    },
    1: {
        "interpretation": "Older, rooted, owner-occupier communities with high outright ownership, high retirement, high White British share and low churn. This is a clear settled older homeowner geography.",
        "confidence": "High — large, coherent and easily interpretable.",
    },
    2: {
        "interpretation": "Professional, family-oriented suburban communities: high owner occupation, houses, married-couple families, Level 4+ qualifications and managerial/professional occupations. This is the core middle-class suburban cluster.",
        "confidence": "High — large and coherent.",
    },
    3: {
        "interpretation": "Young, urban, graduate and cosmopolitan neighbourhoods: high 25–34, flats, private renting, Level 4+ qualifications, non-UK-born and Other White populations. Distinct from student areas and from settled diverse communities.",
        "confidence": "High — distinctive urban professional/transient profile.",
    },
    4: {
        "interpretation": "A broad employed, settled working-family cluster: UK-born, White British, house-based, higher skilled trades/process occupations, Level 1–2 qualifications and moderate owner occupation. It is less extreme than other clusters, but socially meaningful rather than a residual bucket.",
        "confidence": "Medium-high — less statistically extreme, but coherent in profile and likely very important politically.",
    },
    5: {
        "interpretation": "Settled diverse urban communities: high non-white and non-UK-born shares, higher unemployment, mixed residence length, more lone-parent families and less owner occupation. Distinct from the young cosmopolitan professional core.",
        "confidence": "High — clearly interpretable and substantively important.",
    },
    6: {
        "interpretation": "Post-industrial and deprived working communities: high social renting, no qualifications, routine/service/elementary work, long-term sickness/disability and lone-parent households. This is a clearly distinct estate/deprivation profile.",
        "confidence": "High — politically and socially very meaningful.",
    },
}


def top_features(defining_df, cid, direction, n=6):
    part = defining_df[
        (defining_df["cluster_id"] == cid)
        & (defining_df["direction"] == direction)
    ].copy()

    if part.empty:
        return ""

    part["abs_z"] = part["z_score"].abs()
    part = part.sort_values("abs_z", ascending=False).head(n)

    return "; ".join(
        f"{row.feature} ({row.z_score:+.2f})"
        for _, row in part.iterrows()
    )


def age_profile_from_row(row):
    bands = [
        ("0–14", row["age_0_14_count"]),
        ("15–24", row["age_15_24_count"]),
        ("25–34", row["age_25_34_count"]),
        ("35–49", row["age_35_49_count"]),
        ("50–64", row["age_50_64_count"]),
        ("65+", row["age_65_plus_count"]),
    ]

    top2 = sorted(bands, key=lambda x: x[1], reverse=True)[:2]
    return f"Largest age bands: {top2[0][0]} ({pct(top2[0][1])}) and {top2[1][0]} ({pct(top2[1][1])})."


def means_text(row, cols):
    parts = []
    for label, col in cols:
        if col in row.index:
            parts.append(f"{label} {pct(row[col])}")
    return ", ".join(parts)


rows = []

for _, srow in summaries.sort_values("cluster_id").iterrows():
    cid = int(srow["cluster_id"])
    mrow = means.loc[means["cluster_id"] == cid].iloc[0]
    arow = age.loc[age["cluster_id"] == cid].iloc[0]

    rows.append({
        "cluster_id": cid,
        "cluster_name": CLUSTER_NAMES.get(cid, f"Cluster {cid}"),
        "population": int(srow["population"]),
        "population_share": srow["population_share"],
        "oa_count": int(srow["oa_count"]),
        "oa_share": srow["oa_share"],
        "top_high_features": top_features(defining, cid, "high", 6),
        "top_low_features": top_features(defining, cid, "low", 6),
        "age_profile": age_profile_from_row(arow),
        "housing_profile": means_text(mrow, [
            ("owner-occupied", "owned_pct"),
            ("owned outright", "owns_outright_pct"),
            ("social rented", "social_rented_pct"),
            ("private rented", "private_rented_pct"),
            ("houses/bungalows", "house_type_pct"),
            ("flats", "flat_type_pct"),
        ]),
        "occupation_profile": means_text(mrow, [
            ("managerial/professional", "managerial_professional_pct"),
            ("skilled/traditional", "skilled_traditional_pct"),
            ("routine/service/elementary", "routine_service_elementary_pct"),
            ("employed", "employed_pct"),
            ("unemployed", "unemployed_pct"),
            ("student", "full_time_student_pct"),
            ("retired", "retired_pct"),
            ("long-term sick/disabled", "long_term_sick_disabled_pct"),
        ]),
        "education_profile": means_text(mrow, [
            ("no qualifications", "no_qualifications_pct"),
            ("Level 1–2", "level_1_2_pct"),
            ("apprenticeship", "apprenticeship_pct"),
            ("Level 4+", "level_4_plus_pct"),
        ]),
        "rootedness_profile": means_text(mrow, [
            ("UK-born", "uk_born_pct"),
            ("non-UK-born", "non_uk_born_pct"),
            ("10+ years residence", "resident_10_plus_years_pct"),
            ("under 5 years residence", "resident_less_5_years_pct"),
            ("White British", "white_british_pct"),
            ("non-white", "non_white_pct"),
        ]),
        "interpretation": PROFILE_TEXT.get(cid, {}).get("interpretation", ""),
        "confidence": PROFILE_TEXT.get(cid, {}).get("confidence", "Review"),
    })

cluster_key = pd.DataFrame(rows)

cluster_key_path = OUTPUT_DIR / f"k{K}_cluster_interpretation_key_v1.csv"
cluster_key.to_csv(cluster_key_path, index=False)

print("Saved:", cluster_key_path.name)
display(cluster_key)

Saved: k7_cluster_interpretation_key_v1.csv


,cluster_id,cluster_name,population,population_share,oa_count,oa_share,top_high_features,top_low_features,age_profile,housing_profile,occupation_profile,education_profile,rootedness_profile,interpretation,confidence
0,0,Student & Transient Youth,1515395,0.025427,3544,0.018763,full_time_student_pct (+5.40); age_15_24_pct (...,age_50_64_pct (-1.95); level_1_2_pct (-1.81); ...,Largest age bands: 15–24 (53.9%) and 25–34 (14...,"owner-occupied 30.0%, owned outright 15.9%, so...","managerial/professional 47.6%, skilled/traditi...","no qualifications 9.1%, Level 1–2 12.0%, appre...","UK-born 69.9%, non-UK-born 30.1%, 10+ years re...",A small but very sharply defined student and s...,"High — very distinctive, though small."
1,1,Rooted Older Homeowners,12288654,0.206193,42860,0.226917,retired_pct (+1.23); age_65_plus_pct (+1.22); ...,age_25_34_pct (-0.74); non_uk_born_pct (-0.70)...,Largest age bands: 65+ (32.6%) and 50–64 (23.0%).,"owner-occupied 81.6%, owned outright 53.2%, so...","managerial/professional 45.5%, skilled/traditi...","no qualifications 19.0%, Level 1–2 23.7%, appr...","UK-born 94.7%, non-UK-born 5.3%, 10+ years res...","Older, rooted, owner-occupier communities with...","High — large, coherent and easily interpretable."
2,2,Stable Suburban Professionals,11713258,0.196539,35337,0.187087,married_couple_family_pct (+1.12); managerial_...,routine_service_elementary_pct (-1.01); no_qua...,Largest age bands: 50–64 (22.3%) and 35–49 (20...,"owner-occupied 83.1%, owned outright 43.0%, so...","managerial/professional 62.3%, skilled/traditi...","no qualifications 10.4%, Level 1–2 20.4%, appr...","UK-born 88.3%, non-UK-born 11.7%, 10+ years re...","Professional, family-oriented suburban communi...",High — large and coherent.
3,3,Cosmopolitan Young Professional Core,4737944,0.079499,16223,0.085891,white_other_pct (+1.88); age_25_34_pct (+1.82)...,house_type_pct (-1.79); level_1_2_pct (-1.57);...,Largest age bands: 25–34 (26.1%) and 35–49 (24...,"owner-occupied 39.0%, owned outright 16.2%, so...","managerial/professional 64.8%, skilled/traditi...","no qualifications 10.6%, Level 1–2 13.5%, appr...","UK-born 60.5%, non-UK-born 39.5%, 10+ years re...","Young, urban, graduate and cosmopolitan neighb...",High — distinctive urban professional/transien...
4,4,Settled Working Families / Skilled Trades Suburbs,13169130,0.220967,40911,0.216598,employed_pct (+0.60); level_1_2_pct (+0.58); s...,level_4_plus_pct (-0.39); age_65_plus_pct (-0....,Largest age bands: 35–49 (20.6%) and 50–64 (19...,"owner-occupied 64.5%, owned outright 29.3%, so...","managerial/professional 39.2%, skilled/traditi...","no qualifications 17.8%, Level 1–2 26.8%, appr...","UK-born 89.2%, non-UK-born 10.8%, 10+ years re...","A broad employed, settled working-family clust...","Medium-high — less statistically extreme, but ..."
5,5,Settled Diverse Urban Communities,7687366,0.128988,20664,0.109403,non_white_pct (+2.09); resident_10_plus_years_...,white_british_pct (-1.98); uk_born_pct (-1.73)...,Largest age bands: 35–49 (22.1%) and 0–14 (22....,"owner-occupied 39.3%, owned outright 18.5%, so...","managerial/professional 34.6%, skilled/traditi...","no qualifications 24.6%, Level 1–2 22.8%, appr...","UK-born 57.7%, non-UK-born 42.3%, 10+ years re...",Settled diverse urban communities: high non-wh...,High — clearly interpretable and substantively...
6,6,Post-Industrial Estates / Deprived Working Com...,8486000,0.142388,29341,0.155342,long_term_sick_disabled_pct (+1.57); social_re...,managerial_professional_pct (-1.14); owned_pct...,Largest age bands: 0–14 (20.2%) and 50–64 (18....,"owner-occupied 35.6%, owned outright 18.2%, so...","managerial/professional 26.4%, skilled/traditi...","no qualifications 29.5%, Level 1–2 27.8%, appr...","UK-born 89.3%, non-UK-born 10.7%, 10+ years re...",Post-industrial and deprived working communiti...,High — politically and socially very meaningful.


## 10. Final output list

In [12]:
outputs = sorted(OUTPUT_DIR.glob("*.csv"))

print("Generated / available CSVs in aggregations_v1:")
for path in outputs:
    print(path.name)

Generated / available CSVs in aggregations_v1:
k7_cluster_interpretation_key_v1.csv
k7_lad23_cluster_profile_v1.csv
k7_lad23_full_profile_v1.csv
k7_lad23_named_full_profile_v1.csv
k7_lad23_stats_profile_v1.csv
k7_lad25_cluster_profile_v1.csv
k7_lad25_full_profile_v1.csv
k7_lad25_named_full_profile_v1.csv
k7_lad25_stats_profile_v1.csv
k7_lsoa21_cluster_profile_v1.csv
k7_lsoa21_full_profile_v1.csv
k7_lsoa21_named_full_profile_v1.csv
k7_lsoa21_stats_profile_v1.csv
k7_lsoa_named_full_profile_v1.csv
k7_msoa21_cluster_profile_v1.csv
k7_msoa21_full_profile_v1.csv
k7_msoa21_map_ready_v1.csv
k7_msoa21_named_full_profile_v1.csv
k7_msoa21_stats_profile_v1.csv
k7_msoa_named_full_profile_v1.csv
k7_north_west_ward25_full_profile_v1.csv
k7_oa_geo_cluster_base_v1.csv
k7_oa_geo_coverage_report_v1.csv
k7_ward25_cluster_profile_v1.csv
k7_ward25_full_profile_v1.csv
k7_ward25_map_ready_v1.csv
k7_ward25_named_full_profile_v1.csv
k7_ward25_named_full_profile_v1_enriched.csv
k7_ward25_named_full_profile_v1_en